# Pre-Wave 5 — Easing into Amazon Bedrock (demo notebook)
### Connect → list models → say hello → grow it, one cell at a time

**Runs end to end with zero AWS credentials.** It ships with a deterministic mock of the two Bedrock
clients (`bedrock` and `bedrock-runtime`) that returns the same response shapes the real APIs return —
so the whole spiral runs offline, even if your account quota is still being sorted.

To run it for real: set `USE_BEDROCK = True` in the config cell. The *same code* then calls real Bedrock.

> Goal: by the last cell you have listed models, run your first inference, read the cost, streamed a reply,
> hit the inference-profile wall on purpose, and inspected a model — so the Day 5 build feels easy.

## Two clients, one spiral

```
                       USE_BEDROCK ?
                 yes  /             \  no (default)
       real boto3 clients            mock clients (no creds, same shapes)

  get_bedrock()  -> "bedrock"          MANAGEMENT plane: list_foundation_models, get_foundation_model
  get_runtime()  -> "bedrock-runtime"  RUNTIME plane:    converse, converse_stream, invoke_model
```

The climb: **list models → converse hello → read usage → + system/temperature → + multi-turn →
+ streaming → + robustness → the inference-profile wall → inspect models.** Each cell changes one thing.

**Prereqs:** mock mode needs only Python 3.9+. Real mode needs `pip install boto3`, credentials, and Bedrock access.

In [1]:
import json, itertools

# ============================ CONFIG ============================
USE_BEDROCK = False          # flip to True for the real Bedrock APIs
REGION      = "us-east-1"

# works whether or not botocore is installed (mock mode needs nothing):
try:
    from botocore.exceptions import ClientError
except Exception:
    class ClientError(Exception):
        pass

USAGE_LOG = []   # mock calls record (input_tokens, output_tokens) so we can show a cost ledger

def get_bedrock():
    """MANAGEMENT client: lists & describes models. Does NOT run them."""
    if USE_BEDROCK:
        import boto3
        return boto3.client("bedrock", region_name=REGION)
    return MockBedrock()

def get_runtime():
    """RUNTIME client: runs models. adaptive retries back off the throttling you may have seen."""
    if USE_BEDROCK:
        import boto3
        from botocore.config import Config
        return boto3.client("bedrock-runtime", region_name=REGION,
                            config=Config(retries={"max_attempts": 4, "mode": "adaptive"}))
    return MockRuntime()

def cost_report(in_rate=3.0, out_rate=15.0):
    ti = sum(a for a, _ in USAGE_LOG); to = sum(b for _, b in USAGE_LOG)
    cost = ti/1e6*in_rate + to/1e6*out_rate
    print(f"calls={len(USAGE_LOG)}  input_tokens={ti}  output_tokens={to}  est_cost=${cost:.4f}")
    print("(rates illustrative $/1M tokens; confirm current Bedrock pricing. Real-mode calls aren't logged here.)")

print("config loaded - USE_BEDROCK =", USE_BEDROCK)


config loaded - USE_BEDROCK = False


In [2]:
# ===================================================================
#  MOCK ENGINE - the only reason this runs without AWS. Returns the
#  SAME shapes the real bedrock / bedrock-runtime clients return.
#  Read once, then forget it exists.
# ===================================================================
_PREFIXES = ("us.", "eu.", "apac.", "jp.", "global.")

_CATALOG = [
    {"modelId":"anthropic.claude-opus-4-7",  "modelName":"Claude Opus 4.7",  "providerName":"Anthropic",
     "inferenceTypesSupported":["ON_DEMAND","INFERENCE_PROFILE"], "inputModalities":["TEXT","IMAGE"],
     "outputModalities":["TEXT"], "responseStreamingSupported":True},
    {"modelId":"anthropic.claude-sonnet-4-6", "modelName":"Claude Sonnet 4.6","providerName":"Anthropic",
     "inferenceTypesSupported":["INFERENCE_PROFILE"], "inputModalities":["TEXT","IMAGE"],
     "outputModalities":["TEXT"], "responseStreamingSupported":True},
    {"modelId":"anthropic.claude-haiku-4-5",  "modelName":"Claude Haiku 4.5", "providerName":"Anthropic",
     "inferenceTypesSupported":["INFERENCE_PROFILE"], "inputModalities":["TEXT","IMAGE"],
     "outputModalities":["TEXT"], "responseStreamingSupported":True},
    {"modelId":"amazon.nova-2-lite",          "modelName":"Nova 2 Lite",      "providerName":"Amazon",
     "inferenceTypesSupported":["INFERENCE_PROFILE"], "inputModalities":["TEXT","IMAGE"],
     "outputModalities":["TEXT"], "responseStreamingSupported":True},
    {"modelId":"meta.llama3-1-70b-instruct-v1:0","modelName":"Llama 3.1 70B Instruct","providerName":"Meta",
     "inferenceTypesSupported":["ON_DEMAND"], "inputModalities":["TEXT"], "outputModalities":["TEXT"],
     "responseStreamingSupported":True},
    {"modelId":"mistral.mistral-large-2407-v1:0","modelName":"Mistral Large","providerName":"Mistral AI",
     "inferenceTypesSupported":["ON_DEMAND"], "inputModalities":["TEXT"], "outputModalities":["TEXT"],
     "responseStreamingSupported":True},
]
for _m in _CATALOG:
    _m["modelArn"] = f"arn:aws:bedrock:{REGION}::foundation-model/{_m['modelId']}"
    _m["modelLifecycle"] = {"status": "ACTIVE"}

def _match(base):
    for m in _CATALOG:
        if base == m["modelId"] or base.startswith(m["modelId"]):
            return m
    return None
def _strip_prefix(model_id):
    for p in _PREFIXES:
        if model_id.startswith(p):
            return model_id[len(p):], True
    return model_id, False
def _approx(s): return max(1, len(str(s))//4)
def _last_user(messages):
    for m in reversed(messages):
        if m["role"] == "user":
            for b in m["content"]:
                if "text" in b: return b["text"]
    return ""

class MockValidation(Exception): pass

def _reply(messages):
    t = _last_user(messages).lower()
    if "hello" in t:                     return "Hello! Amazon Bedrock runs many foundation models behind one Converse API."
    if "beach" in t:                     return "Consider Goa - relaxed beaches and easy flights from most Indian cities."
    if "trip" in t and "budget" in t:    return "Budget 3-day Goa: hostel stays, local buses, beach shacks for food."
    if "trip" in t:                      return "A 3-day trip works well: two beach days and one for exploring old Goa."
    if "pitch" in t or "goa" in t:       return "Sun-soaked Goa in 3 days: beaches by day, night markets after dark."
    return "Understood. (mock reply - set USE_BEDROCK=True for the real model.)"

class MockRuntime:
    def _guard(self, modelId):
        base, has_prefix = _strip_prefix(modelId)
        rec = _match(base)
        if (not has_prefix) and rec and "ON_DEMAND" not in rec["inferenceTypesSupported"]:
            raise MockValidation(
                f"An error occurred (ValidationException) when calling the Converse operation: "
                f"Invocation of model ID {modelId} with on-demand throughput isn't supported. "
                f"Retry your request with the ID or ARN of an inference profile that contains this model.")
    def converse(self, modelId, messages, system=None, inferenceConfig=None, toolConfig=None, guardrailConfig=None):
        self._guard(modelId)
        text = _reply(messages); usage = {"inputTokens":_approx(messages), "outputTokens":_approx(text)}
        usage["totalTokens"] = usage["inputTokens"]+usage["outputTokens"]
        USAGE_LOG.append((usage["inputTokens"], usage["outputTokens"]))
        return {"output":{"message":{"role":"assistant","content":[{"text":text}]}},
                "stopReason":"end_turn", "usage":usage}
    def converse_stream(self, modelId, messages, system=None, inferenceConfig=None, **kw):
        self._guard(modelId)
        text = _reply(messages)
        USAGE_LOG.append((_approx(messages), _approx(text)))
        def gen():
            for w in text.split(" "):
                yield {"contentBlockDelta":{"delta":{"text": w+" "}}}
            yield {"messageStop":{"stopReason":"end_turn"}}
        return {"stream": gen()}
    def invoke_model(self, modelId, body):
        self._guard(modelId)
        req = json.loads(body); msgs = req.get("messages", [])
        last = msgs[-1]["content"] if msgs else ""
        text = _reply([{"role":"user","content":[{"text": last if isinstance(last,str) else ""}]}])
        USAGE_LOG.append((_approx(body), _approx(text)))
        payload = {"id":"msg_mock","type":"message","role":"assistant",
                   "content":[{"type":"text","text":text}], "stop_reason":"end_turn",
                   "usage":{"input_tokens":_approx(body),"output_tokens":_approx(text)}}
        class _Body:
            def __init__(self,d): self._d=d
            def read(self): return json.dumps(self._d).encode()
        return {"body": _Body(payload)}

class MockBedrock:
    def list_foundation_models(self, **kw):
        return {"modelSummaries":[dict(m) for m in _CATALOG]}
    def get_foundation_model(self, modelIdentifier):
        base,_ = _strip_prefix(modelIdentifier); rec = _match(base) or _CATALOG[0]
        return {"modelDetails": dict(rec)}

print("mock engine ready")


mock engine ready


---
## Part A - List the models (are we connected?)

The **management** client. No model is run, no tokens spent. If it returns model IDs, your wiring is correct.

In [3]:
bedrock = get_bedrock()                                   # MANAGEMENT client

models = bedrock.list_foundation_models()["modelSummaries"]
for m in models[:6]:
    print(f'{m["modelId"]:<34} | {m["providerName"]:<10} | {",".join(m["inferenceTypesSupported"])}')


anthropic.claude-opus-4-7          | Anthropic  | ON_DEMAND,INFERENCE_PROFILE
anthropic.claude-sonnet-4-6        | Anthropic  | INFERENCE_PROFILE
anthropic.claude-haiku-4-5         | Anthropic  | INFERENCE_PROFILE
amazon.nova-2-lite                 | Amazon     | INFERENCE_PROFILE
meta.llama3-1-70b-instruct-v1:0    | Meta       | ON_DEMAND
mistral.mistral-large-2407-v1:0    | Mistral AI | ON_DEMAND


---
## Part B - Your first inference, then grow it

Switch to the **runtime** client and send a prompt. Then read the response by walking this tree:

```
resp
└─ "output"
   └─ "message"
      ├─ "role"      ->  "assistant"
      └─ "content"
         └─ [0]
            └─ "text" ->  the model's reply
resp["usage"]        ->  {inputTokens, outputTokens, totalTokens}
resp["stopReason"]   ->  'end_turn' | 'max_tokens' | 'tool_use'
```

In [4]:
brt = get_runtime()                                      # RUNTIME client
MODEL_ID = "anthropic.claude-opus-4-7"   # base ID runs on-demand in us-east-1 (no profile needed yet)

conversation = [{"role":"user", "content":[{"text":"Say hello and name one thing Bedrock does."}]}]
try:
    resp = brt.converse(modelId=MODEL_ID, messages=conversation,
                        inferenceConfig={"maxTokens":200, "temperature":0.5})
    print(resp["output"]["message"]["content"][0]["text"])     # walk: output > message > content > [0] > text
except (ClientError, Exception) as e:
    print(f"ERROR: cannot invoke {MODEL_ID}. Reason: {e}")


Hello! Amazon Bedrock runs many foundation models behind one Converse API.


In [5]:
print("usage     :", resp["usage"])        # tokens in/out/total - read this from your FIRST call
print("stopReason:", resp["stopReason"])   # why it ended; 'max_tokens' means you were cut off


usage     : {'inputTokens': 21, 'outputTokens': 18, 'totalTokens': 39}
stopReason: end_turn


In [6]:
SYSTEM = [{"text": "You are TravelMind's assistant. Be concise. Never invent prices."}]
resp = brt.converse(
    modelId=MODEL_ID,
    system=SYSTEM,                                          # behavioural contract goes HERE, not the user turn
    messages=[{"role":"user","content":[{"text":"Suggest a beach city in India."}]}],
    inferenceConfig={"maxTokens":150, "temperature":0},     # 0 = repeatable, ideal for parsing/tests
)
print(resp["output"]["message"]["content"][0]["text"])


Consider Goa - relaxed beaches and easy flights from most Indian cities.


In [7]:
messages = [{"role":"user","content":[{"text":"I want a 3-day trip."}]}]
r1 = brt.converse(modelId=MODEL_ID, messages=messages, inferenceConfig={"maxTokens":200})
messages.append(r1["output"]["message"])                                  # keep the model's reply
messages.append({"role":"user","content":[{"text":"Make it budget-friendly."}]})   # follow-up
r2 = brt.converse(modelId=MODEL_ID, messages=messages, inferenceConfig={"maxTokens":200})
print(r2["output"]["message"]["content"][0]["text"])
print("\n(memory = this growing messages list; the API itself is stateless)")


Understood. (mock reply - set USE_BEDROCK=True for the real model.)

(memory = this growing messages list; the API itself is stateless)


### Stream the reply

`converse_stream` returns an event stream; you assemble the `contentBlockDelta` pieces as they arrive.

In [8]:
stream = brt.converse_stream(
    modelId=MODEL_ID,
    messages=[{"role":"user","content":[{"text":"Write a 2-line pitch for Goa."}]}],
    inferenceConfig={"maxTokens":200},
)
for chunk in stream["stream"]:
    if "contentBlockDelta" in chunk:
        print(chunk["contentBlockDelta"]["delta"]["text"], end="")
print()


Sun-soaked Goa in 3 days: beaches by day, night markets after dark. 


### Make it robust

Three habits separate a notebook that worked once from a service that runs at 2 a.m.

1. **adaptive retries** on the client (already set in `get_runtime`) - backs off `ThrottlingException`.
2. **always set `maxTokens`** - Bedrock reserves quota against the model's max output; an unbounded reply is an unbounded bill, and can even trip a daily limit.
3. **catch `ClientError`** - credentials, throttling, validation all surface there.

> The *"Too many tokens per day"* throttling you saw earlier was a different beast: an account-level **quota** sitting near zero, not your code. Mock mode lets you keep building regardless.

In [9]:
def ask(brt, model_id, prompt, max_tokens=200):
    try:
        r = brt.converse(modelId=model_id,
                         messages=[{"role":"user","content":[{"text":prompt}]}],
                         inferenceConfig={"maxTokens":max_tokens, "temperature":0.3})
        return r["output"]["message"]["content"][0]["text"], r["usage"]
    except (ClientError, Exception) as e:
        return f"[handled] {e}", None

text, usage = ask(brt, MODEL_ID, "Give one tip for a first-time Goa traveller.")
print(text, "\n", usage)


Sun-soaked Goa in 3 days: beaches by day, night markets after dark. 
 {'inputTokens': 22, 'outputTokens': 16, 'totalTokens': 38}


### The inference-profile wall (on purpose)

Switch to a cheaper, profile-only model with the **bare** ID and it fails - then the `us.` prefix fixes it.
This is the exact thing Day 5 unpacks. (In mock mode the same rule is simulated, so it runs offline.)

In [10]:
cheap_bare   = "anthropic.claude-haiku-4-5"                       # profile-only -> bare ID rejected
cheap_profile = "us.anthropic.claude-haiku-4-5-20251001-v1:0"     # the prefix = a cross-region profile

try:
    brt.converse(modelId=cheap_bare,
                 messages=[{"role":"user","content":[{"text":"hi"}]}],
                 inferenceConfig={"maxTokens":50})
except (ClientError, Exception) as e:
    print("FAILED (bare ID):", str(e)[:130], "...\n")

r = brt.converse(modelId=cheap_profile,
                 messages=[{"role":"user","content":[{"text":"hi"}]}],
                 inferenceConfig={"maxTokens":50})
print("WORKS (with us. profile):", r["output"]["message"]["content"][0]["text"])


FAILED (bare ID): An error occurred (ValidationException) when calling the Converse operation: Invocation of model ID anthropic.claude-haiku-4-5 wit ...

WORKS (with us. profile): Understood. (mock reply - set USE_BEDROCK=True for the real model.)


---
## Part C - Discover & inspect models

Two single management calls: list what's available, then read one model's fine print.

In [11]:
claude = [m for m in bedrock.list_foundation_models()["modelSummaries"]
          if m["providerName"] == "Anthropic"]
for m in claude:
    print(f'{m["modelId"]:<30} -> {",".join(m["inferenceTypesSupported"])}')   # ON_DEMAND? or profile-only?


anthropic.claude-opus-4-7      -> ON_DEMAND,INFERENCE_PROFILE
anthropic.claude-sonnet-4-6    -> INFERENCE_PROFILE
anthropic.claude-haiku-4-5     -> INFERENCE_PROFILE


In [12]:
d = bedrock.get_foundation_model(modelIdentifier="anthropic.claude-haiku-4-5")["modelDetails"]
print("name      :", d["modelName"], "|", d["providerName"])
print("input     :", d["inputModalities"])          # can it take images?
print("output    :", d["outputModalities"])
print("inference :", d["inferenceTypesSupported"])  # this field tells you bare-ID vs profile
print("streaming :", d["responseStreamingSupported"])


name      : Claude Haiku 4.5 | Anthropic
input     : ['TEXT', 'IMAGE']
output    : ['TEXT']
inference : ['INFERENCE_PROFILE']
streaming : True


---
## Part D - the other door: Invoke (provider-native)

Converse is one shape across every model. `invoke_model` is the raw, provider-specific body - use it only
when you need a feature Converse hasn't exposed. Shown once so you recognise it.

In [13]:
raw = brt.invoke_model(
    modelId="anthropic.claude-opus-4-7",
    body=json.dumps({
        "anthropic_version": "bedrock-2023-05-31",     # provider-specific envelope
        "messages": [{"role":"user","content":"Hello from Invoke"}],
        "max_tokens": 200,
    }),
)
print(json.loads(raw["body"].read())["content"][0]["text"])
print("\n(Converse hides this envelope. Switch models under Invoke = rewrite the body; under Converse = one string.)")


Hello! Amazon Bedrock runs many foundation models behind one Converse API.

(Converse hides this envelope. Switch models under Invoke = rewrite the body; under Converse = one string.)


---
## Wrap - you're ready for Day 5

You connected, listed models, ran inference, read cost, streamed, hit the profile wall, and inspected a model.

**Day 5 builds straight on this:** the same `converse` call becomes a **workflow** (messy text -> forced JSON)
and an **agent** (the model picks tools in a loop). Same client, same API.

**To go live:** set `USE_BEDROCK = True`, confirm your quota is non-zero, use the `us.` profile for most Claude
models, and read `usage` on every call.

In [14]:
cost_report()

calls=8  input_tokens=200  output_tokens=135  est_cost=$0.0026
(rates illustrative $/1M tokens; confirm current Bedrock pricing. Real-mode calls aren't logged here.)
